------------------
```markdown
# Copyright © 2024 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amandments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com
```
------------------------------
❗❗❗ **IMPORTANT**❗❗❗ **Create a copy of this notebook**

In order to work with this Google Colab you need to create a copy of it. Please **DO NOT** provide your answers here. Instead, work on the copy version. To make a copy:

**Click on: File -> save a copy in drive**

Have you successfully created the copy? if yes, there must be a new tab opened in your browser. Now move to the copy and start from there!

----------------------------------------------


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Introduction to the Datasets
We analyze the data from an experiment that evaluates the effectiveness of a program that offers teachers in Italian middle schools the chance to participate in a training. The training is designed to help them promote inclusive practices in the classroom. The experiment was designed as a randomized control trial (RCT), however we will not enter the RCT-based analysis. Here, we will clean the data from this experiment and perform some basic analysis.


In [ ]:
# Load the first dataset. Here is the link to the dataset:
# https://drive.google.com/uc?id=1oE-3rt17bFW7fOzDIjwFSEMPTIV3NvcO

df = pd.read_csv("https://drive.google.com/uc?id=1oE-3rt17bFW7fOzDIjwFSEMPTIV3NvcO")

See what columns are there. Try to make sense of them.

In [ ]:
df.columns
df.describe()

Is there any column with strange values?

##### Student Dataset:
Contain administrative information on students’ characteristics and academic performance. The data are organized at the student level and contain the following variables:
- **school_id** : A variable that identifies in which school the student is enrolled.
- **grade**: A variable that identifies students’ grade.
- **class**: A variable that identifies students’ class. In our setting, students are enrolled in only one school-grade-class.
- **student_id** : A variable containing an unique identifier for students (their email address).
- **sex**: A variable that identifies students’ sex. 1 refers to female students and 2 to male students.
- **nationality**: A variable that identifies students’ nationality. 1 refers to students with Italian citizenship and 2 to foreign students.
- **grade_math_t1** : Students’ grade in math at period 1 (i.e. before the intervention)
- **grade_language_t1** : Students’ grade in language at period 1 (i.e. before the intervention)
- **grade_science_t1** : Students’ grade in science at period 1 (i.e. before the intervention)
- **grade_math_t2** : Students’ grade in math at period 2 (i.e. after the intervention)
- **grade_language_t2** : Students’ grade in language at period 2 (i.e. after the intervention)
- **grade_science_t2** : Students’ grade in science at period 2 (i.e. after the intervention)
- **treatment**: A variable indicating if the student was taught by a teacher who was assigned to the treatment group.

**Note**: The italian grade system grades students from 0 to 10.

#### Example
Perform the following tasks.

* Extract information about columns' datatype and shape.

In [ ]:
df.shape
df.dtypes

* Rename all the grade columns such that the part "grade_" is removed from the column name.

In [ ]:
df = df.rename(columns=lambda x: x.replace('grade_', ''))
# df = df.columns.str.replace('grade_', '')
# df = df.rename(columns=lambda x: x.replace("grade_", ""))
df.columns

* Figure out if there are duplicates.

In [ ]:
df.nunique()
# df.loc[df["student_id"].duplicated(), "student_id"]

**Quiz**: Do you know how to compute the number of duplicates for each column?

In [ ]:
# df["count"] = df["student_id"].duplicated().sum()
df.apply(lambda col: col.duplicated().sum())

**Challenge:** How would you handle the duplicates?

In [ ]:
df = df.copy().drop_duplicates()


#### Exercise 1
Write a function that iterates through certain columns and determines whether all the entries of those columns unique. Use the following template.

In [ ]:
def _unique_check(df: pd.DataFrame, cols: list) -> list:
    """Verifies that entries in the input columns are unique.

        Args:
          df: A DataFrame.
          cols: List of columns names to check.

        Returns:
          A list with the structure ["unique_check", "col", "PASSED"/"FAILED"].
    """
    result = []
    for col in cols:
        if df[col].is_unique:
            result.append(["unique_check", col, "PASSED"])
        else:
            result.append(["unique_check", col, "FAILED"])
    return result

result = _unique_check(df=df, cols=list(df.columns))
print(result)

**Question**: What about the missing values? How do we handle them?

In [ ]:
df.isna().sum()

#### Exercise 2
Write a function that iterates through certain columns and determines whether they contain missing values or not. Use the following template

In [ ]:
def _missing_value_check(df: pd.DataFrame, cols: list) -> list:
    """Used to verify that the input column in the DataFrame does not contain missing values.

    Args:
        df: The DataFrame containing the data.
        cols: List of columns names to check.

    Returns:
        A list with the structure ["missing_value_check", "col", "PASSED"/"FAILED"] for each column.
    """
    result = []
    for col in cols:
        if df[col].isnull().any(): # df[col] = [1, 2, None, 8] --> [False, False, True, False]
            result.append(["missing_value_check", col, "FAILED"])
        else:
            result.append(["missing_value_check", col, "PASSED"])

    # for col in cols:
    #     # Your code
    #     if df[col].isna().sum()!=0: # df[col] = [1, 2, None, 8] --> [False, False, True, False]
    #       result.append(['missing_value_check', col, 'FAILED'])
    #     else:
    #       result.append(['missing_value_check', col, 'PASSED'])
    return result

result = _missing_value_check(df=df, cols=["student_id", "language_t1", "science_t2"])
print(result)


# How to deal with the missing data?
In general, missing values could be random, or related to certain aspect of the data. **An example of the latter**: in the income surveys, individual with higher income are less willing to report their income.

General ways to handle missing values:

- remove the corresponding row or column
- imputation - mean, median, mode, regression etc.
- Flagging - using a flag to indicate the missing value

**BUT** we also need to identify the values that are missing but not explicitly designated by **null**, **NaN** or **None**. In the case of our dataset, the grades greater than 10.

**Attention**: Is there any other case/column to be considered?


-----------------------
#### Exercise 3
We know from the properties of the dataset that the student grades cannot be greater than 10. Write a function to check if the values of the grade columns are less than 10 or not.

In [ ]:
def _upper_limit_check(df: pd.DataFrame, cols: list, limit: float) -> list:
    """Verifies that entries in the input columns are less than a certain limit.

    Args:
        df: The DataFrame containing the data.
        cols: List of column names to check.
        limit: Upper limit for the values.

    Returns:
        A list with the structure ["upper_limit_check", "col", "PASSED"/"FAILED"] for each column.
    """
    result = []
    for col in cols:
        if (df[col] <= limit).all():
          result.append(["upper_limit_check", col, "PASSED"])
        else:
          # df_limit = df.loc[(df[col] > limit), col]
          # print(df_limit)
          result.append(["upper_limit_check", col, "FAILED"])
    return result

result = _upper_limit_check(df=df, cols=["language_t1", "science_t2", "school_id"], limit=10.0)
print(result)

:::Now that you have checked the values, try to handle those greater than 10.

In [1]:
# Replace the values greater than 10 in the grade columns with np.nan.
df.loc[:, "math_t1":"science_t2"] = df.loc[:, "math_t1":"science_t2"].apply(
    lambda x: np.where(x > 10, np.nan, x)
)

# df.loc[:, "math_t1":"science_t2"] = np.where(
    # df.loc[:, "math_t1":"science_t2"]>10, np.nan, df.loc[:, "math_t1":"science_t2"]
# )
df.isna().sum()

NameError: name 'df' is not defined

In [1]:
# Handle the missing values.

df["math_t2_fillna"] = df.groupby(by=["school_id", "grade", "class"])["math_t2"].transform("mean")
print(df["math_t2"][df["math_t2"].isna()])
# print(df[df["math_t2"].isna(), "math_t2"])
print(df["math_t2_fillna"][df["math_t2"].isna()])
df.loc[df["math_t2"].isna(), "math_t2"] = df.loc[df["math_t2"].isna(), "math_t2_fillna"]
df = df.drop(columns=["math_t2_fillna"], axis=1)

df["language_t2_fillna"] = df.groupby(by=["school_id", "grade", "class"])["language_t2"].transform("mean")
print(df["language_t2"][df["language_t2"].isna()])
print(df["language_t2_fillna"][df["language_t2"].isna()])
df.loc[df["language_t2"].isna(), "language_t2"] = df.loc[df["language_t2"].isna(), "language_t2_fillna"]
df = df.drop(columns=["language_t2_fillna"], axis=1)

df["science_t2_fillna"] = df.groupby(by=["school_id", "grade", "class"])["science_t2"].transform("mean")
print(df["science_t2"][df["science_t2"].isna()])
print(df["science_t2_fillna"][df["science_t2"].isna()])
df.loc[df["science_t2"].isna(), "science_t2"] = df.loc[df["science_t2"].isna(), "science_t2_fillna"]
df = df.drop(columns=["science_t2_fillna"], axis=1)

NameError: name 'df' is not defined


**Question**: How should we address the *date_of_birth* column?

<!--
# Copyright © 2024 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amandments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com.
-->

#### Exercise 4
Write a function that checks if all the date formats of the a date column, e.g. *date_of_birth* is correct.

In [ ]:
def _date_check(df: pd.DataFrame, col: str) -> list:
    """Used to verify that the column in the input DataFrame has a correct format for converting to datetime.

    Args:
        df: DataFrame containing input data.
        col: Name of column which is checked.

    Returns:
        A list with the structure ["_date_check", "col", "PASSED"/"FAILED"] for each column.
    """
    results = []
    try:
        # Convert the column to datetime and see if any error raised, if yes,
        # there is a problem with the format
        pd.to_datetime(df[col], errors='raise')
        results.append(["date_check", col, "PASSED"])
    except ValueError:
        results.append(["date_check", col, "FAILED"])

    return results

result = _date_check(df=df, col="date_of_birth")
print(result)

**Challenge**: Is there anyway to identify the non-conforming entries without looking for them in the dataset.

#### Exercise 5
- Import the survey_data.csv using the link:
https://drive.google.com/uc?id=1A0B4z0Dohs8gfikJyh4p_FEUWyPD2U9f
- Repeat all the above steps.
- For each survey question i (variables q1 to q8), create a numeric variable called qib (i.e., q1b to q8b) that assigns a numeric value to each qualitative answer.
- Merge the two datasets. **Hint**: Think about the column that is present in both datasets and is a primary key.

<!--
# Copyright © 2024 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amandments:
- Individual may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com.
-->

In [ ]:
df_survey = pd.read_csv("https://drive.google.com/uc?id=1A0B4z0Dohs8gfikJyh4p_FEUWyPD2U9f")

 See what columns are there. Try to make sense of them.

In [ ]:
df_survey.head(5)

##### Survey Dataset
Contain information from a survey that was sent to students before the intervention. Participation in the survey was optional. These data are organized at the student level and contain the following variables:
- **school_id** : A variable that identifies in which school the student is enrolled.
- **grade**: A variable that identifies students’ grade.
- **class**: A variable that identifies students’ class. In our setting, students are enrolled in only
one schoolgrade- class.
- **student_id** : A variable containing an unique identifier for students (their email address).
- **q1** : Survey question 1 ”Would you say you enjoy studying mathematics?”.
- **q2** : Survey question 2 ”Would you say you enjoy studying Language?”.
- **q3** : Survey question 3 ”Would you say you enjoy studying science?”.
- **q4** : Survey question 4 ”Would you agree with the following statement: I believe my teacher
cares about my learning.”.
- **q5** : Survey question 5 ”Would you agree with the following statement: I believe my teacher
cares about my well-being.”.
- **q6** : Survey question 6 ”Would you agree with the following statement: I believe my teacher
helps create an inclusive environment in the classroom.”.
- **q7** : Survey question 7 ”Would you agree with the following statement: I believe my school
provides adequate resources for underrepresented students.”.
- **q8** : Survey question 8 ”What are your plans for the future?”.



Perform the following tasks:

* Extract information about columns' datatype and shape.

In [ ]:
df_survey.shape
df_survey.dtypes

* Figure out if there are duplicates. If yes, how many in each column?

In [ ]:
df_survey.nunique()

num_duplicates = df_survey.apply(lambda x: x.duplicated().sum())
print(num_duplicates)

* Figure about the null/missing values.

In [ ]:
df_survey.isna().sum()

* Handle the duplicates and the missing values. Then check again the if there are missing values.

In [1]:
# Handle the duplicates and the missing values.
df_survey.loc[:, "q1": "q8"] = df_survey.loc[:, "q1": "q8"].apply(lambda x: np.where(x.isna(), x.mode(), x))
df_survey.isna().sum()


NameError: name 'df_survey' is not defined

* Merge the survey dataset with the previous one using left join

In [ ]:
df_tot = df.merge(df_survey, how="left", on=["student_id", "school_id", "grade", "class"])
df_tot.columns
df_tot.shape

* Merge the survey dataset with the previous one using inner join

In [ ]:
df_tot = df.merge(df_survey, how="inner", on=["student_id"])
df_tot.columns
df_tot.shape

#### Exercise 6
#### data cleaning using object-oriented programming
Assume that the following table is given to you. We call it **coldef table** (column definition table), describing what checks must be run for each column to ensure that the entries are clean. 0 (or empty cell) indicates irrelvence of the property for the column while 1 denotes that the check must be run for the column in question.

 column name | missing_value_check | unique_check | numeric | upper_limit | date
 :- | -: | :-: |  :-: | :-: | :-:
  student_id | 1 | 1 |  | |
  math_t1 | 1 | 0 | 1 | 10 | |
  math_t2 | 1 | 0 | 1 | 10 | |
  date_of_birth | | | | | %Y-%d-%m

  #### Tasks:
  1. Define a new dataframe which consists of only the columns in the above table, i.e., a slice of the originial dataframe.
  2. Use the following object-oriented programming structure to check whether all the columns pass the checks mentioned in the table. A part of the code is written to help you figure out the rest.

In [ ]:
coldef = pd.DataFrame(
    {
        "column_name": ["student_id", "math_t1", "math_t2", "date_of_birth"],
        "missing_value_check": [1, 1, 1, np.nan],
        "unique_check": [1, 0, 0, np.nan],
        "numeric": [0, 1, 1, np.nan],
        "upper_limit": [np.nan, 10, 10, np.nan],
        "date": [None, None, None, "%Y-%m-%d"],
    }
)
data = df[["student_id", "math_t1", "math_t2", "date_of_birth"]]

class ColdefCheck:
  def __init__(self, data: pd.DataFrame, coldef: pd.DataFrame):
    self.coldef = coldef
    self.data = data

  def execute_coldef_checks(self):
    """Used to execute the different coldef checks and generate an overview of the results.

        Returns:
            A list of all results.
    """
    results = []
    results += self._missing_value_check()
    results += self._unique_check()
    results += self._numeric_check()
    results += self._upper_limit_check()
    results += self._date_check()
    return results

  def _missing_value_check(self) -> list:
    """Used to verify that the input column in the DataFrame does not contain missing values.

        Returns:
            A list with the structure ["missing_value_check", "col", "PASSED"/"FAILED"].
    """
    result = []
    for col in self.coldef.loc[self.coldef["missing_value_check"]==1, "column_name"]:
        if df[col].isnull().any(): # df[col] = [1, 2, None, 8] --> [False, False, True, False]
            result.append(["missing_value_check", col, "FAILED"])
        else:
            result.append(["missing_value_check", col, "PASSED"])

    return result

  def _unique_check(self) -> list:
    """Verifies that entries in the input columns are unique.

        Returns:
            A list with the structure ["unique_check", "col", "PASSED"/"FAILED"].
    """
    result = []
    for col in self.coldef.loc[self.coldef["unique_check"]==1, "column_name"]:
        if df[col].is_unique:
            result.append(["unique_check", col, "PASSED"])
        else:
            result.append(["unique_check", col, "FAILED"])
    return result

  def _numeric_check(self) -> list:
    """Verifies that the columns in the input DataFrame has a numeric ``dtype``.

        Returns:
            A list with the structure ["numeric_check",  "col", "PASSED"/"FAILED"].
    """
    result = []
    for col in self.coldef.loc[self.coldef["numeric"] == 1, "column_name"]:
      if self.data[col].dtype in ["int", "float"]:
        result.append(["numeric_check", col,  "PASSED"])
      else:
        result.append(["numeric_check", col, "FAILED"])
    return result

  def _upper_limit_check(self) -> list:
    """Verifies that entries in the input columns are less than a certain limit.

        Returns:
            A list with the structure ["upper_limit_check", "col", "PASSED"/"FAILED"].
    """
    result = []
    for col in self.coldef.loc[self.coldef["upper_limit"] > 0, "column_name"]:
      if (df[col] <= self.coldef.loc[self.coldef["column_name"]==col, "upper_limit"].iloc[0]).all():
        result.append(["upper_limit_check", col, "PASSED"])
      else:
        # df_limit = df.loc[(df[col] > limit), col]
        # print(df_limit)
        result.append(["upper_limit_check", col, "FAILED"])
    return result

  def _date_check(self) -> list:
    """Used to verify that the column in the input DataFrame has a correct format for converting to datetime.

    Args:
        df: DataFrame containing input data.
        col: Name of column which is checked.

    Returns:
        A list with the structure ["_date_check", "col", "PASSED"/"FAILED"] for each column.
    """
    results = []
    for col in self.coldef.loc[self.coldef["date"].notna(), "column_name"]:
      try:
          # Convert the column to datetime and see if any error raised, if yes,
          # there is a problem with the format
          date_format = self.coldef.loc[self.coldef["column_name"]==col, "date"].iloc[0]
          pd.to_datetime(df["date_of_birth"], format=date_format, errors="raise")
          results.append(["date_check", col, "PASSED"])
          # Your code
      except ValueError:
          results.append(["date_check", col, "FAILED"])

    return results

# We define an instance of our class (my version of the blueprint :-D)
my_dataset = ColdefCheck(data=data, coldef=coldef)

# To access any variable of function in my version of the class, I can use the
# my_version_of_class.variable or my_version_of_class.function
results = my_dataset.execute_coldef_checks()
for result in results:
  print(result)


**Congratulations! You have finished the Notebook! Great Job!**
🤗🙌👍👏💪
<!--
# Copyright © 2024 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amandments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com.
-->